In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"C:\Users\Raunak Verma\Downloads\archive\Telco-Customer-Churn.csv")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors ='coerce')

In [ ]:
df = df.dropna(subset=['TotalCharges'])

In [ ]:
df.isnull().sum()

In [ ]:
# Drop the customerID column

df = df.drop(['customerID'], axis=1)

# EDA

## Churn Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# count churn value

print(df['Churn'].value_counts())

In [ ]:
# plot churn distribution

sns.countplot(x='Churn', data=df, hue='Churn', legend=False, palette='Set2')
plt.title("Customer Churn Distribution")
plt.show()


In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df, palette='Set2')
plt.title("Churn by Contract Type")
plt.show()


In [ ]:
sns.countplot(x='tenure', hue='Churn', data=df, palette='Set2')
plt.title("Churn by Tenure")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', palette='Set2')
plt.title("Churn by Tenure")
plt.xlabel("Customer Tenure (Months)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Churn', y='MonthlyCharges', hue='Churn', legend=False, data=df, palette='Set2')
plt.title("Monthly Charges vs. Churn")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Churn', y='TotalCharges', hue='Churn', legend=False, data=df, palette='Set2')
plt.title("Total Charges vs Churn")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='tenure', y='MonthlyCharges', data=df)
plt.title("Tenure vs Monthly Charges")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='tenure', y='MonthlyCharges', data=df, hue='Churn', palette='Set2')
plt.title("Tenure vs Monthly Charges (colored by Churn)")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
corr = df[['tenure','MonthlyCharges','TotalCharges']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


# Step 4 Data Preprocessing for Modeling

## Step 4.1 Handling Missing Values.

In [ ]:
df.isnull().sum()

## Step 4.2 — Encoding Categorical Variables

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

## Step 4.3 - Scaling Numeric Features

In [ ]:
# Step 1: Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 2: Handle missing values (if any blanks turned into NaN)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Step 3: Encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Step 4: Scale numeric features
from sklearn.preprocessing import StandardScaler
num_cols = ['tenure','MonthlyCharges','TotalCharges']
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

# Step 5: Verify
df_encoded[num_cols].head()

In [ ]:
from sklearn.preprocessing import StandardScaler
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols]) #
df_encoded[num_cols].head()

## Step 4.4 Train_Test_split

In [ ]:
from sklearn.model_selection import train_test_split
X = df_encoded.drop('Churn_Yes', axis=1) #features
y = df_encoded['Churn_Yes'] #Target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set:", X_test.shape, y_test.shape)

In [ ]:
# Show first 20 columns
df_encoded.columns[:20]

# Search for churn-related columns
[col for col in df_encoded.columns if "Churn" in col]


In [ ]:
df_encoded.columns[:20]

In [ ]:
[col for col in df_encoded.columns if "Churn" in col]
[col for col in df_encoded.columns if "InternetService" in col]
[col for col in df_encoded.columns if "PaymentMethod" in col]


## Step 5.1 Logistic Regression (baseline model)

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred= log_reg.predict(X_test)
y_pred_prob= log_reg.predict_proba(X_test)[:,1]

In [ ]:
print(y_pred[:10])

# Compare predictions with actual labels
print("Predicted:", y_pred[:10])
print("Actual   :", y_test[:10].values)

## Step 5.2 Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

#Accuracy

print("Accuracy:", accuracy_score(y_test, y_pred))

#confusion Matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

#classification report
print("Classification Report:\n", classification_report(y_test, y_pred))

## Step 5.3 Improving the Model

### Step A Scaling Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Step B: Handling Class Imbalance

In [ ]:
log_reg_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
log_reg_balanced.fit(X_train_scaled, y_train)
y_pred_balanced = log_reg_balanced.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

# Confusion Matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))

# Classification Report
print("Classification Report:\n", classification_report(y_test, y_pred_balanced))


## Step 5.4 Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,   #number of tree

    random_state=42, #reproducibility

    class_weight='balanced'  #handel imbalanced
)

In [ ]:
# Fitting the model

rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


## Step 5.5: XGBoost model

In [ ]:
%pip install xgboost

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# Step 0: Import libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Step 1: Load dataset
df = pd.read_csv(r"C:\Users\Raunak Verma\Downloads\archive\Telco-Customer-Churn.csv")

# Step 2: Clean 'TotalCharges' column (convert to numeric and fill missing)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Step 3: Convert target column 'Churn' into 0/1
df['Churn'] = df['Churn'].map({'No':0, 'Yes':1})

# Step 4: Encode categorical variables (except numeric ones)
df_encoded = pd.get_dummies(df, drop_first=True)

# Step 5: Scale numeric features
num_cols = ['tenure','MonthlyCharges','TotalCharges']
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

# Step 6: Split into train/test
X = df_encoded.drop('Churn', axis=1)   # features
y = df_encoded['Churn']                # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,  # numbers of tree
    learning_rate=0.1,   #step size for boosting
    max_depth=5,            # depth of each tree
    random_state=42,        # reproducibility
    scale_pos_weight=3      # handle imbalance (ratio of non-churners to churners)
)

In [ ]:
# Fit the model
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


## Step 6: Feature Importance in XGBoos

In [ ]:
# Step 1: Import and define the model
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    scale_pos_weight=3
)

# Step 2: Fit the model
xgb_model.fit(X_train, y_train)

# Step 3: Get feature importance
importance = xgb_model.feature_importances_
features = X_train.columns

# Step 4: Plot
plt.figure(figsize=(10,6))
plt.barh(features, importance)
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("XGBoost Feature Importance for Churn Prediction")
plt.show()

In [ ]:
import pandas as pd

# Get importance scores
importance = xgb_model.feature_importances_
features = X_train.columns

# Create a DataFrame
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importance})

# Sort by importance
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

# Show top 10
print(feat_imp.head(10))

# Plot top 10
feat_imp.head(10).plot(kind='barh', x='Feature', y='Importance', figsize=(8,6), legend=False)


## Problem Statement

Customer churn is a critical challenge for subscription-based businesses. High churn rates directly reduce revenue and customer lifetime value.  
The goal of this project is to build a predictive model to identify at-risk customers and uncover the key drivers of churn, enabling targeted retention strategies.

The top churn drivers are:
- Contract type (short contracts increase churn risk)  
- Internet service (fiber optic users show higher churn tendency)  
- Tenure (new customers are more likely to churn)  
- Payment method (electronic check users churn more often)  

**Business Recommendations:**
- Encourage longer contracts through loyalty discounts or bundles.  
- Focus retention campaigns on new customers in their first few months.  
- Monitor fiber optic customers closely and improve service satisfaction.  
- Promote auto-payment methods to reduce churn risk.  

By acting on these insights, the business can reduce churn significantly, protect revenue, and improve customer loyalty.



## Key Results

Our churn prediction model was trained using XGBoost and evaluated on a held‑out test set.

- **Accuracy:** ~75%  
- **Recall (Churners):** ~75%  
- **Precision (Churners):** ~70%  

These results show that the model is effective at identifying customers at risk of churn, with strong recall ensuring most churners are correctly flagged.

### Top Features Driving Churn
The most influential features identified by the model are:
1. **Contract type** – Two‑year contracts strongly reduce churn risk; month‑to‑month contracts increase risk.  
2. **Internet service** – Fiber optic customers show higher churn tendency compared to DSL.  
3. **Tenure** – New customers are more likely to churn; longer tenure reduces risk.  
4. **Payment method** – Electronic check users churn more often.  
5. **Add‑on services** – Streaming and security services reduce churn likelihood.

These findings highlight the customer segments most at risk and provide clear signals for targeted retention strategies.


## Insights & Recommendations

Our churn prediction analysis highlights clear business drivers:

- **Contract type** is the strongest predictor. Customers on month‑to‑month contracts are far more likely to churn, while one‑year and two‑year contracts reduce churn risk significantly.  
- **Internet service type** matters. Fiber optic customers show higher churn tendency compared to DSL or no service.  
- **Tenure** confirms the onboarding challenge. New customers are more likely to churn, while longer tenure reduces risk.  
- **Payment method** is a churn signal. Customers paying by electronic check churn more often.  
- **Add‑on services** (streaming, online security) reduce churn likelihood, suggesting bundled services improve retention.

### Business Recommendations
- **Encourage longer contracts** by offering loyalty discounts, bundles, or perks for one‑year and two‑year plans.  
- **Focus retention campaigns on new customers** during their first few months to build loyalty early.  
- **Monitor fiber optic customers closely** and improve satisfaction through service quality initiatives.  
- **Promote auto‑payment methods** (credit card, bank transfer) to reduce churn risk.  
- **Bundle add‑on services** like streaming or security to increase customer stickiness.

### Conclusion
By acting on these insights, the business can reduce churn significantly, protect recurring revenue, and strengthen customer loyalty.  
This project demonstrates how machine learning can translate raw data into actionable strategies that directly impact business outcomes.
